# Text2Concept — Real Training on CIFAR-10 (Google Colab)

This notebook trains a `Text2ConceptAligner` on CIFAR-10 using a pretrained ResNet-50 backbone,
then demonstrates all three paper applications:
1. Concept similarity (sort images by text concept)
2. Zero-shot classification
3. Distribution shift diagnosis

## Cell 1 — Runtime check

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Cell 2 — Install

In [ ]:
!pip install text2concept-pytorch torchinfo -q

## Cell 3 — Download CIFAR-10 and build paired dataset

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torchvision import models
from torch.utils.data import DataLoader, Dataset
import open_clip
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from text2concept_pytorch import Text2Concept, Text2ConceptAligner, IMAGENET_TEMPLATES
from text2concept_pytorch.text2concept_pytorch import l2norm, flatten_features

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# CLIP preprocessing
_, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')

# Source encoder preprocessing (standard ImageNet stats)
source_transform = T.Compose([
    T.Resize(224),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class PairedCIFAR10(Dataset):
    """CIFAR-10 returning (source_transform(img), clip_transform(img), label)."""
    def __init__(self, root, train=True):
        self.base = torchvision.datasets.CIFAR10(root=root, train=train, download=True)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        img = img if isinstance(img, Image.Image) else Image.fromarray(img)
        return source_transform(img), clip_preprocess(img), label

train_ds = PairedCIFAR10('./data', train=True)
test_ds  = PairedCIFAR10('./data', train=False)

train_dl = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

print(f'Train: {len(train_ds):,} images | Test: {len(test_ds):,} images')

## Cell 4 — Build encoder (ResNet-50, frozen)

In [ ]:
# ResNet-50 pretrained on ImageNet, classification head removed
backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
backbone.fc = nn.Identity()
backbone = backbone.to(DEVICE).eval()
for p in backbone.parameters():
    p.requires_grad = False

SOURCE_DIM = 2048
CLIP_DIM   = 512
print(f'Encoder output dim: {SOURCE_DIM}')

## Cell 5 — Build CLIP model and aligner

In [ ]:
clip_model, _, _ = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
clip_model = clip_model.to(DEVICE).eval()
for p in clip_model.parameters():
    p.requires_grad = False

tokenizer = open_clip.get_tokenizer('ViT-B-16')

aligner = Text2ConceptAligner(source_dim=SOURCE_DIM, clip_dim=CLIP_DIM).to(DEVICE)
optimizer = torch.optim.AdamW(aligner.parameters(), lr=1e-3)
print('Aligner parameters:', sum(p.numel() for p in aligner.parameters()))

## Cell 6 — Training loop (30 epochs)

In [ ]:
from tqdm.notebook import tqdm

NUM_EPOCHS  = 30
epoch_losses = []

for epoch in range(NUM_EPOCHS):
    aligner.train()
    total_loss  = 0.
    num_batches = 0

    for source_imgs, clip_imgs, _ in tqdm(train_dl, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}', leave=False):
        source_imgs = source_imgs.to(DEVICE)
        clip_imgs   = clip_imgs.to(DEVICE)

        with torch.no_grad():
            source_feats = backbone(source_imgs)                     # (b, 2048)
            clip_feats   = l2norm(clip_model.encode_image(clip_imgs)) # (b, 512)

        aligned = aligner(source_feats)                              # (b, 512)
        loss    = (1 - (aligned * clip_feats).sum(dim=-1)).mean()    # cosine loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss  += loss.item()
        num_batches += 1

    avg = total_loss / num_batches
    epoch_losses.append(avg)
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | cosine loss: {avg:.4f}')

# Save checkpoint
import os
os.makedirs('./results', exist_ok=True)
torch.save({
    'aligner'    : aligner.state_dict(),
    'source_dim' : aligner.source_dim,
    'clip_dim'   : aligner.clip_dim,
}, './results/aligner-final.pt')
print('Saved to ./results/aligner-final.pt')

## Cell 7 — Training curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, NUM_EPOCHS + 1), epoch_losses, marker='o', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Cosine Loss')
plt.title('Text2ConceptAligner — Training Loss (CIFAR-10)')
plt.grid(True)
plt.tight_layout()
plt.savefig('./results/training_curve.png', dpi=150)
plt.show()
print(f'Final loss: {epoch_losses[-1]:.4f}')

## Cell 8 — Application 1: Zero-shot classification on test set

In [ ]:
# Build Text2Concept from the trained aligner
aligner_loaded = Text2ConceptAligner.from_checkpoint('./results/aligner-final.pt', device=DEVICE)
t2c = Text2Concept(backbone, aligner_loaded).to(DEVICE)

# Encode class name vectors once
class_vecs = torch.cat([
    t2c.encode_concept(cls) for cls in CIFAR10_CLASSES
], dim=0)  # (10, 512)

# Evaluate zero-shot accuracy
all_preds, all_labels = [], []
aligner_loaded.eval()

with torch.no_grad():
    for source_imgs, _, labels in tqdm(test_dl, desc='Zero-shot eval'):
        source_imgs = source_imgs.to(DEVICE)
        feats       = l2norm(aligner_loaded(backbone(source_imgs)))  # (b, 512)
        sims        = feats @ class_vecs.T                           # (b, 10)
        preds       = sims.argmax(dim=-1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds)
all_labels = torch.cat(all_labels)
accuracy   = (all_preds == all_labels).float().mean().item() * 100
print(f'Zero-shot accuracy on CIFAR-10 test set: {accuracy:.2f}%')

## Cell 9 — Per-class accuracy

In [ ]:
per_class_acc = []
for i, cls in enumerate(CIFAR10_CLASSES):
    mask = all_labels == i
    acc  = (all_preds[mask] == i).float().mean().item() * 100
    per_class_acc.append(acc)
    print(f'{cls:>12s}: {acc:.1f}%')

plt.figure(figsize=(10, 4))
bars = plt.bar(CIFAR10_CLASSES, per_class_acc, color='steelblue')
plt.axhline(accuracy, color='red', linestyle='--', label=f'Mean {accuracy:.1f}%')
plt.ylabel('Accuracy (%)')
plt.title('Zero-shot Classification Accuracy per Class')
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('./results/per_class_accuracy.png', dpi=150)
plt.show()

## Cell 10 — Application 2: Concept similarity — sort images by concept

In [ ]:
# Grab a batch of test images (raw PIL for display)
raw_test = torchvision.datasets.CIFAR10('./data', train=False, download=False)
raw_imgs_pil = [raw_test[i][0] for i in range(200)]
raw_labels   = [raw_test[i][1] for i in range(200)]

# Preprocess for encoder
batch_tensor = torch.stack([source_transform(img) for img in raw_imgs_pil]).to(DEVICE)

CONCEPT = 'animal'
with torch.no_grad():
    sims = t2c.concept_similarity(batch_tensor, [CONCEPT]).squeeze(1).cpu()  # (200,)

# Sort by similarity
sorted_idx = sims.argsort(descending=True)

def show_top_bottom(concept, sorted_idx, raw_imgs, labels, k=8):
    fig, axes = plt.subplots(2, k, figsize=(16, 5))
    fig.suptitle(f'Concept: "{concept}"', fontsize=14)
    for i, idx in enumerate(sorted_idx[:k]):
        axes[0, i].imshow(raw_imgs[idx])
        axes[0, i].set_title(CIFAR10_CLASSES[labels[idx]], fontsize=8)
        axes[0, i].axis('off')
    for i, idx in enumerate(sorted_idx[-k:]):
        axes[1, i].imshow(raw_imgs[idx])
        axes[1, i].set_title(CIFAR10_CLASSES[labels[idx]], fontsize=8)
        axes[1, i].axis('off')
    axes[0, 0].set_ylabel('Most similar', fontsize=9)
    axes[1, 0].set_ylabel('Least similar', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'./results/concept_{concept.replace(" ", "_")}.png', dpi=150)
    plt.show()

show_top_bottom(CONCEPT, sorted_idx, raw_imgs_pil, raw_labels)

## Cell 11 — Application 3: Distribution shift diagnosis

In [ ]:
from scipy.stats import ks_2samp

# Simulate distribution shift: compare 'vehicle' concept similarity
# between animal classes (bird, cat, deer, dog, frog, horse) and vehicle classes
# (airplane, automobile, ship, truck)
ANIMAL_IDXS  = [2, 3, 4, 5, 6, 7]   # bird, cat, deer, dog, frog, horse
VEHICLE_IDXS = [0, 1, 8, 9]          # airplane, automobile, ship, truck

concept = 'vehicle'
with torch.no_grad():
    all_sims = t2c.concept_similarity(batch_tensor, [concept]).squeeze(1).cpu().numpy()

animal_sims  = all_sims[[i for i, l in enumerate(raw_labels) if l in ANIMAL_IDXS]]
vehicle_sims = all_sims[[i for i, l in enumerate(raw_labels) if l in VEHICLE_IDXS]]

stat, p = ks_2samp(animal_sims, vehicle_sims)
print(f'KS statistic: {stat:.4f}  |  p-value: {p:.4e}')
print('Significant shift detected!' if p < 0.05 else 'No significant shift.')

plt.figure(figsize=(8, 4))
plt.hist(animal_sims,  bins=20, alpha=0.6, label='Animal classes',  color='green')
plt.hist(vehicle_sims, bins=20, alpha=0.6, label='Vehicle classes', color='steelblue')
plt.xlabel(f'Cosine similarity to "{concept}"')
plt.ylabel('Count')
plt.title(f'Distribution Shift Diagnosis — concept: "{concept}"\nKS stat={stat:.3f}, p={p:.2e}')
plt.legend()
plt.tight_layout()
plt.savefig('./results/distribution_shift.png', dpi=150)
plt.show()

## Cell 12 — Summary of results

In [ ]:
print('=' * 45)
print('  Text2Concept — CIFAR-10 Results Summary')
print('=' * 45)
print(f'  Final cosine loss     : {epoch_losses[-1]:.4f}')
print(f'  Zero-shot accuracy    : {accuracy:.2f}%')
print(f'  KS stat (shift diag.) : {stat:.4f} (p={p:.2e})')
print('=' * 45)
print('Saved outputs:')
print('  results/aligner-final.pt')
print('  results/training_curve.png')
print('  results/per_class_accuracy.png')
print('  results/concept_animal.png')
print('  results/distribution_shift.png')